# Minimum SAE Circuit Discovery v002: Wanda-Style Ranking and Sanity Checks

This Colab notebook extends the MVP magnitude baseline with checks and baselines motivated by the PDF review:

1. Sanity-check SAE reconstruction faithfulness across layers 6, 7, 8, and 9.
2. Compare plain SAE substitution against error-preserving SAE feature ablation.
3. Add a zero-mask baseline and larger K values up to the full SAE width.
4. Add Wanda-style feature ranking: decoder feature norm times activation norm.
5. Add final-token feature ranking to test feature-position sensitivity for IOI.

The main expected improvement over v001 is `preserve_error=True`: the model activation is decomposed into SAE reconstruction plus reconstruction error, and feature masks only edit the SAE component. With all features kept, this returns the original activation exactly, avoiding the low all-feature reconstruction ceiling from v001.


## 1. Colab Setup

Run this notebook in a GPU runtime. The smoke cells are small; the full baseline comparison runs several top-K sweeps and is intended for Colab Pro+.


In [ ]:
%pip install -q "sae-lens>=6,<7" pandas matplotlib tqdm


In [ ]:
import json
import os
import random
import shutil
from collections import defaultdict
from contextlib import nullcontext
from datetime import datetime
from functools import partial
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import torch
from sae_lens import SAE, HookedSAETransformer
from tqdm.auto import tqdm

SEED = 12345
random.seed(SEED)
torch.manual_seed(SEED)

if not torch.cuda.is_available():
    raise RuntimeError("This notebook expects a CUDA GPU. In Colab, enable Runtime -> Change runtime type -> GPU.")

device = "cuda"
torch.set_grad_enabled(False)

props = torch.cuda.get_device_properties(0)
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {props.total_memory / 1024**3:.1f} GB")
print(f"PyTorch: {torch.__version__}")


## 2. Versioned Drive Output

Outputs are saved to Google Drive under `MyDrive/minimum_sae_circuit_discovery`. Each run has a versioned trial folder, shared caches are versioned separately, and completed artifacts are mirrored to `latest/<RUN_VERSION>/`.


In [ ]:
MOUNT_DRIVE = True
PROJECT_DIR_NAME = "minimum_sae_circuit_discovery"
RUN_VERSION = "v002_wanda_position_sanity"
TRIAL_ID = os.environ.get("TRIAL_ID") or datetime.now().strftime("trial_%Y%m%d_%H%M%S")
RUN_STARTED_AT = datetime.now().astimezone().isoformat(timespec="seconds")

try:
    from google.colab import drive  # type: ignore
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB and MOUNT_DRIVE:
    drive.mount("/content/drive")
    PROJECT_ROOT = Path("/content/drive/MyDrive") / PROJECT_DIR_NAME
else:
    PROJECT_ROOT = Path.cwd() / f"{PROJECT_DIR_NAME}_outputs"

CACHE_DIR = PROJECT_ROOT / "cache" / RUN_VERSION
OUTPUT_DIR = PROJECT_ROOT / "runs" / RUN_VERSION / TRIAL_ID
LATEST_DIR = PROJECT_ROOT / "latest" / RUN_VERSION
for directory in (CACHE_DIR, OUTPUT_DIR, LATEST_DIR):
    directory.mkdir(parents=True, exist_ok=True)

MODEL_NAME = "gpt2-small"
SAE_RELEASE = "gpt2-small-res-jb"
TARGET_LAYER = 8
LAYER_SWEEP = [6, 7, 8, 9]
EXPECTED_D_SAE = 24_576

SMOKE_N_PROMPTS = 8
N_PROMPTS = 1_000
LAYER_SANITY_N_PROMPTS = 128
BATCH_SIZE = 16
K_VALUES = [5, 10, 20, 50, 100, 200, 500, 1_000, 2_000, 5_000, 10_000, 20_000, EXPECTED_D_SAE]
SMOKE_K_VALUES = [5, 20, 100, 1_000]
PRESERVE_ERROR_FOR_SWEEP = True

STATS_CACHE_PATH = CACHE_DIR / f"feature_stats_layer{TARGET_LAYER}_ioi.pt"
SMOKE_STATS_CACHE_PATH = CACHE_DIR / f"smoke_feature_stats_layer{TARGET_LAYER}_ioi.pt"
LAYER_SANITY_CSV_PATH = OUTPUT_DIR / "layer_reconstruction_sanity.csv"
SMOKE_RESULTS_CSV_PATH = OUTPUT_DIR / "smoke_baseline_comparison_results.csv"
RESULTS_CSV_PATH = OUTPUT_DIR / "baseline_comparison_results.csv"
SMOKE_PLOT_PATH = OUTPUT_DIR / "smoke_baseline_comparison.png"
PLOT_PATH = OUTPUT_DIR / "baseline_comparison.png"
MANIFEST_PATH = OUTPUT_DIR / "run_manifest.json"


def hook_name_for_layer(layer):
    return f"blocks.{layer}.hook_resid_pre"


def sae_id_for_layer(layer):
    return hook_name_for_layer(layer)


def write_run_manifest(status, extra=None):
    manifest = {
        "status": status,
        "run_started_at": RUN_STARTED_AT,
        "run_updated_at": datetime.now().astimezone().isoformat(timespec="seconds"),
        "run_version": RUN_VERSION,
        "trial_id": TRIAL_ID,
        "seed": SEED,
        "model_name": MODEL_NAME,
        "sae_release": SAE_RELEASE,
        "target_layer": TARGET_LAYER,
        "layer_sweep": LAYER_SWEEP,
        "n_prompts": N_PROMPTS,
        "smoke_n_prompts": SMOKE_N_PROMPTS,
        "layer_sanity_n_prompts": LAYER_SANITY_N_PROMPTS,
        "batch_size": BATCH_SIZE,
        "k_values": K_VALUES,
        "preserve_error_for_sweep": PRESERVE_ERROR_FOR_SWEEP,
        "project_root": str(PROJECT_ROOT),
        "cache_dir": str(CACHE_DIR),
        "output_dir": str(OUTPUT_DIR),
        "latest_dir": str(LATEST_DIR),
        "artifacts": {
            "stats_cache_path": str(STATS_CACHE_PATH),
            "layer_sanity_csv_path": str(LAYER_SANITY_CSV_PATH),
            "smoke_results_csv_path": str(SMOKE_RESULTS_CSV_PATH),
            "smoke_plot_path": str(SMOKE_PLOT_PATH),
            "results_csv_path": str(RESULTS_CSV_PATH),
            "plot_path": str(PLOT_PATH),
        },
        "extra": extra or {},
    }
    with open(MANIFEST_PATH, "w", encoding="utf-8") as f:
        json.dump(manifest, f, indent=2)
    return manifest


def mirror_artifacts_to_latest(paths):
    LATEST_DIR.mkdir(parents=True, exist_ok=True)
    copied = []
    for artifact_path in paths:
        artifact_path = Path(artifact_path)
        if artifact_path.exists():
            destination = LATEST_DIR / artifact_path.name
            shutil.copy2(artifact_path, destination)
            copied.append(destination)
    return copied

write_run_manifest("initialized")
print(f"Project root: {PROJECT_ROOT}")
print(f"Run version: {RUN_VERSION}")
print(f"Trial ID: {TRIAL_ID}")
print(f"Output folder: {OUTPUT_DIR}")
print(f"Cache folder: {CACHE_DIR}")


## 3. Load Model and SAEs

The target SAE is layer 8. Additional SAEs for layers 6, 7, and 9 are loaded only when the layer sanity check runs.


In [ ]:
SAE_CACHE = {}


def load_sae_for_layer(layer):
    if layer in SAE_CACHE:
        return SAE_CACHE[layer]
    sae = SAE.from_pretrained(
        release=SAE_RELEASE,
        sae_id=sae_id_for_layer(layer),
        device=device,
    )
    sae.eval()
    for param in sae.parameters():
        param.requires_grad_(False)
    assert sae.cfg.d_sae == EXPECTED_D_SAE, f"Layer {layer}: expected d_sae={EXPECTED_D_SAE}, got {sae.cfg.d_sae}"
    SAE_CACHE[layer] = sae
    return sae


target_sae = load_sae_for_layer(TARGET_LAYER)
metadata = getattr(target_sae.cfg, "metadata", None)
model_kwargs = getattr(metadata, "model_from_pretrained_kwargs", None) or {}
model = HookedSAETransformer.from_pretrained_no_processing(
    MODEL_NAME,
    device=device,
    **model_kwargs,
)
model.eval()

if getattr(model.tokenizer, "pad_token", None) is None:
    model.tokenizer.pad_token = model.tokenizer.eos_token
model.tokenizer.padding_side = "left"

print(f"Loaded model: {MODEL_NAME}")
print(f"Loaded target SAE: {SAE_RELEASE} / {sae_id_for_layer(TARGET_LAYER)}")
print(f"SAE d_in: {target_sae.cfg.d_in}")
print(f"SAE d_sae: {target_sae.cfg.d_sae}")


## 4. IOI-Style Dataset

The dataset is generated inside the notebook so the run is portable from GitHub to Colab. The clean prompt is evaluated at the final token, comparing the indirect-object answer token against the subject answer token.


In [ ]:
NAMES = [
    "John", "Mary", "Bob", "Alice", "Tom", "Sarah", "James", "Emily",
    "Robert", "Laura", "Michael", "Anna", "David", "Lisa", "Daniel", "Emma",
    "Paul", "Karen", "Mark", "Susan", "Peter", "Linda", "Kevin", "Nancy",
    "Steven", "Helen", "George", "Carol", "Brian", "Julia", "Henry", "Megan",
    "Adam", "Rachel", "Patrick", "Olivia", "Andrew", "Grace", "Edward", "Sophie",
]

PLACES = [
    "store", "park", "school", "office", "garden", "library", "station", "market",
    "museum", "theater", "church", "beach", "cafe", "hotel", "airport", "hospital",
]

OBJECTS = [
    "book", "letter", "drink", "snack", "ticket", "phone", "gift", "photo",
    "bag", "card", "key", "toy", "coin", "map", "note", "pen",
]

TEMPLATES = [
    "When {subject} and {io} went to the {place}, {subject} gave a {obj} to",
    "After {subject} and {io} visited the {place}, {subject} handed a {obj} to",
    "While {subject} and {io} waited near the {place}, {subject} passed a {obj} to",
    "Because {subject} and {io} were at the {place}, {subject} offered a {obj} to",
]


def build_ioi_dataset(n_prompts=1_000, seed=0):
    rng = random.Random(seed)
    records = []
    seen = set()
    attempts = 0

    while len(records) < n_prompts and attempts < n_prompts * 100:
        attempts += 1
        subject, io = rng.sample(NAMES, 2)
        place = rng.choice(PLACES)
        obj = rng.choice(OBJECTS)
        template = rng.choice(TEMPLATES)

        clean_prompt = template.format(subject=subject, io=io, place=place, obj=obj)
        corrupt_prompt = template.format(subject=io, io=subject, place=place, obj=obj)
        key = (clean_prompt, corrupt_prompt)
        if key in seen:
            continue
        seen.add(key)

        records.append(
            {
                "clean_prompt": clean_prompt,
                "corrupt_prompt": corrupt_prompt,
                "answer_clean": f" {io}",
                "answer_corrupt": f" {subject}",
                "subject": subject,
                "indirect_object": io,
                "place": place,
                "object": obj,
                "template": template,
            }
        )

    if len(records) < n_prompts:
        raise ValueError(f"Only generated {len(records)} unique prompts out of requested {n_prompts}.")
    return pd.DataFrame(records)


def token_id_for_answer(answer):
    tokens = model.to_tokens(answer, prepend_bos=False).reshape(-1)
    if tokens.numel() != 1:
        pieces = model.to_str_tokens(answer, prepend_bos=False)
        raise ValueError(f"Answer {answer!r} is not a single token: {pieces}")
    return int(tokens.item())


def add_answer_token_ids(dataset):
    dataset = dataset.copy()
    clean_ids = []
    corrupt_ids = []
    for _, row in dataset.iterrows():
        clean_ids.append(token_id_for_answer(row["answer_clean"]))
        corrupt_ids.append(token_id_for_answer(row["answer_corrupt"]))
    dataset["answer_clean_id"] = clean_ids
    dataset["answer_corrupt_id"] = corrupt_ids
    return dataset

smoke_dataset = add_answer_token_ids(build_ioi_dataset(SMOKE_N_PROMPTS, seed=SEED))
full_dataset = add_answer_token_ids(build_ioi_dataset(N_PROMPTS, seed=SEED))
sanity_dataset = full_dataset.head(LAYER_SANITY_N_PROMPTS).copy()

print(smoke_dataset[["clean_prompt", "answer_clean", "answer_corrupt"]].head())
print(f"Smoke prompts: {len(smoke_dataset)}")
print(f"Layer sanity prompts: {len(sanity_dataset)}")
print(f"Full prompts: {len(full_dataset)}")


## 5. Model and SAE Intervention Helpers

`preserve_error=True` is the important v2 change. It returns:

```text
sae.decode(masked_features) + (activation - sae.decode(all_features))
```

With an all-ones mask this exactly recovers the original activation, so top-K curves are not capped by SAE reconstruction error.


In [ ]:
def group_tokenized_texts(texts):
    groups = defaultdict(list)
    for index, text in enumerate(texts):
        tokens = model.to_tokens(text, prepend_bos=True).squeeze(0).to(device)
        groups[int(tokens.numel())].append((index, tokens))
    return groups


def sae_mask_hook(activation, hook, sae, mask=None, preserve_error=True):
    feature_acts = sae.encode(activation)
    full_reconstruction = sae.decode(feature_acts)

    if mask is None:
        masked_feature_acts = feature_acts
    else:
        mask = mask.to(device=feature_acts.device, dtype=feature_acts.dtype).view(1, 1, -1)
        masked_feature_acts = feature_acts * mask

    masked_reconstruction = sae.decode(masked_feature_acts)
    if preserve_error:
        return masked_reconstruction + (activation - full_reconstruction)
    return masked_reconstruction


def run_logits(
    texts,
    batch_size=BATCH_SIZE,
    sae=None,
    layer=None,
    mask=None,
    use_sae_substitution=False,
    preserve_error=True,
):
    if isinstance(texts, str):
        texts = [texts]

    output_logits = [None] * len(texts)
    groups = group_tokenized_texts(texts)

    fwd_hooks = []
    if use_sae_substitution:
        if sae is None or layer is None:
            raise ValueError("sae and layer are required when use_sae_substitution=True")
        fwd_hooks = [
            (
                hook_name_for_layer(layer),
                partial(sae_mask_hook, sae=sae, mask=mask, preserve_error=preserve_error),
            )
        ]

    with torch.no_grad():
        context = model.hooks(fwd_hooks=fwd_hooks) if fwd_hooks else nullcontext()
        with context:
            for items in groups.values():
                for start in range(0, len(items), batch_size):
                    chunk = items[start : start + batch_size]
                    indices = [item[0] for item in chunk]
                    tokens = torch.stack([item[1] for item in chunk], dim=0)
                    logits = model(tokens)
                    final_logits = logits[:, -1, :].detach().cpu()
                    for idx, row_logits in zip(indices, final_logits):
                        output_logits[idx] = row_logits

    return torch.stack(output_logits, dim=0)


def mean_logit_diff(final_logits, dataset):
    clean_ids = torch.tensor(dataset["answer_clean_id"].to_list(), dtype=torch.long)
    corrupt_ids = torch.tensor(dataset["answer_corrupt_id"].to_list(), dtype=torch.long)
    row_indices = torch.arange(len(dataset), dtype=torch.long)
    diffs = final_logits[row_indices, clean_ids] - final_logits[row_indices, corrupt_ids]
    return diffs.mean()


def compute_faithfulness(sae, layer, mask, dataset, full_logit_diff=None, batch_size=BATCH_SIZE, preserve_error=True):
    texts = dataset["clean_prompt"].to_list()

    if full_logit_diff is None:
        full_logits = run_logits(texts, batch_size=batch_size, use_sae_substitution=False)
        full_logit_diff = mean_logit_diff(full_logits, dataset)

    masked_logits = run_logits(
        texts,
        batch_size=batch_size,
        sae=sae,
        layer=layer,
        mask=mask,
        use_sae_substitution=True,
        preserve_error=preserve_error,
    )
    masked_logit_diff = mean_logit_diff(masked_logits, dataset)

    denominator = float(full_logit_diff.item())
    if abs(denominator) < 1e-8:
        raise ValueError(f"Full-model logit diff is too close to zero: {denominator}")

    active_features = int(mask.sum().item()) if mask is not None else int(sae.cfg.d_sae)
    return {
        "faithfulness": float((masked_logit_diff / full_logit_diff).item()),
        "full_logit_diff": float(full_logit_diff.item()),
        "masked_logit_diff": float(masked_logit_diff.item()),
        "active_features": active_features,
    }


def make_topk_mask(top_features, k, d_sae=EXPECTED_D_SAE):
    k = min(int(k), int(d_sae))
    mask = torch.zeros(int(d_sae), device=device)
    mask[top_features[:k].to(device)] = 1.0
    return mask


## 6. Dimension Check

Before running baselines, verify the target SAE dimension matches the layer activation.


In [ ]:
sample_text = smoke_dataset.loc[0, "clean_prompt"]
sample_tokens = model.to_tokens(sample_text, prepend_bos=True).to(device)
_, sample_cache = model.run_with_cache(sample_tokens, names_filter=[hook_name_for_layer(TARGET_LAYER)])
sample_acts = sample_cache[hook_name_for_layer(TARGET_LAYER)]

print(f"Sample activation shape at layer {TARGET_LAYER}: {tuple(sample_acts.shape)}")
print(f"SAE d_in: {target_sae.cfg.d_in}")
print(f"SAE d_sae: {target_sae.cfg.d_sae}")
assert sample_acts.shape[-1] == target_sae.cfg.d_in
assert target_sae.cfg.d_sae == EXPECTED_D_SAE


## 7. Layer Reconstruction Sanity Check

This is the diagnostic missing from v001. It compares all-feature and zero-feature interventions with and without error preservation across layers 6, 7, 8, and 9.


In [ ]:
def run_layer_reconstruction_sanity(dataset, layers=LAYER_SWEEP, batch_size=BATCH_SIZE, output_csv_path=None):
    texts = dataset["clean_prompt"].to_list()
    full_logits = run_logits(texts, batch_size=batch_size, use_sae_substitution=False)
    full_logit_diff = mean_logit_diff(full_logits, dataset)

    rows = []
    for layer in tqdm(layers, desc="Layer sanity"):
        sae = load_sae_for_layer(layer)
        all_mask = torch.ones(int(sae.cfg.d_sae), device=device)
        zero_mask = torch.zeros(int(sae.cfg.d_sae), device=device)
        for preserve_error in [False, True]:
            for mask_name, mask in [("all_features", all_mask), ("zero_features", zero_mask)]:
                metrics = compute_faithfulness(
                    sae=sae,
                    layer=layer,
                    mask=mask,
                    dataset=dataset,
                    full_logit_diff=full_logit_diff,
                    batch_size=batch_size,
                    preserve_error=preserve_error,
                )
                rows.append(
                    {
                        "layer": layer,
                        "mask_name": mask_name,
                        "preserve_error": preserve_error,
                        **metrics,
                    }
                )
    sanity = pd.DataFrame(rows)
    if output_csv_path is not None:
        sanity.to_csv(output_csv_path, index=False)
        print(f"Saved layer sanity results to {output_csv_path}")
    return sanity

layer_sanity = run_layer_reconstruction_sanity(
    sanity_dataset,
    layers=LAYER_SWEEP,
    batch_size=BATCH_SIZE,
    output_csv_path=LAYER_SANITY_CSV_PATH,
)
write_run_manifest("layer_sanity_completed", extra={"layer_sanity_rows": len(layer_sanity)})
layer_sanity


## 8. Cache SAE Feature Statistics

v002 caches more than mean absolute activation:

- all-token mean absolute activation
- final-token mean absolute activation
- all-token L2 activation norm
- final-token L2 activation norm

The L2 norms enable Wanda-style ranking: `decoder_feature_norm * activation_norm`.


In [ ]:
def cache_sae_feature_stats(dataset, sae, layer, cache_path, batch_size=BATCH_SIZE, force_recompute=False):
    cache_path = Path(cache_path)
    if cache_path.exists() and not force_recompute:
        payload = torch.load(cache_path, map_location="cpu")
        print(f"Loaded cached feature stats from {cache_path}")
        return {key: value.to(device) if torch.is_tensor(value) else value for key, value in payload.items()}

    d_sae = int(sae.cfg.d_sae)
    sum_abs_all = torch.zeros(d_sae, device=device)
    sum_abs_final = torch.zeros(d_sae, device=device)
    sum_sq_all = torch.zeros(d_sae, device=device)
    sum_sq_final = torch.zeros(d_sae, device=device)
    total_tokens = 0
    total_prompts = 0

    texts = dataset["clean_prompt"].to_list()
    groups = group_tokenized_texts(texts)
    hook_name = hook_name_for_layer(layer)

    with torch.no_grad():
        for items in tqdm(groups.values(), desc="Token length groups"):
            for start in tqdm(range(0, len(items), batch_size), leave=False, desc="Batches"):
                chunk = items[start : start + batch_size]
                tokens = torch.stack([item[1] for item in chunk], dim=0)
                _, cache = model.run_with_cache(tokens, names_filter=[hook_name])
                acts = cache[hook_name]
                feature_acts = sae.encode(acts)

                abs_features = feature_acts.abs()
                final_features = feature_acts[:, -1, :]
                sum_abs_all += abs_features.sum(dim=(0, 1))
                sum_abs_final += final_features.abs().sum(dim=0)
                sum_sq_all += feature_acts.square().sum(dim=(0, 1))
                sum_sq_final += final_features.square().sum(dim=0)
                total_tokens += feature_acts.shape[0] * feature_acts.shape[1]
                total_prompts += feature_acts.shape[0]

    payload = {
        "mean_abs_all": (sum_abs_all / total_tokens).detach().cpu(),
        "mean_abs_final": (sum_abs_final / total_prompts).detach().cpu(),
        "l2_all": sum_sq_all.sqrt().detach().cpu(),
        "l2_final": sum_sq_final.sqrt().detach().cpu(),
        "rms_all": (sum_sq_all / total_tokens).sqrt().detach().cpu(),
        "rms_final": (sum_sq_final / total_prompts).sqrt().detach().cpu(),
        "total_tokens": int(total_tokens),
        "total_prompts": int(total_prompts),
        "layer": int(layer),
        "hook_name": hook_name,
        "sae_release": SAE_RELEASE,
        "sae_id": sae_id_for_layer(layer),
    }
    torch.save(payload, cache_path)
    print(f"Saved feature stats cache to {cache_path}")
    return {key: value.to(device) if torch.is_tensor(value) else value for key, value in payload.items()}


def get_decoder_feature_norm(sae):
    W_dec = sae.W_dec.detach()
    if W_dec.shape[0] == int(sae.cfg.d_sae):
        return W_dec.norm(dim=1)
    if W_dec.shape[-1] == int(sae.cfg.d_sae):
        return W_dec.norm(dim=0)
    raise ValueError(f"Cannot infer decoder feature axis from W_dec shape {tuple(W_dec.shape)}")


def make_feature_rankings(stats, sae):
    decoder_norm = get_decoder_feature_norm(sae).to(device)
    scores = {
        "activation_mean_abs_all": stats["mean_abs_all"],
        "wanda_decoder_l2_all": decoder_norm * stats["l2_all"],
        "final_token_mean_abs": stats["mean_abs_final"],
        "final_token_wanda": decoder_norm * stats["l2_final"],
    }
    rankings = {name: score.argsort(descending=True) for name, score in scores.items()}
    return scores, rankings


## 9. Baseline Comparison Helpers

The comparison runs four rankings over the same K values:

1. `activation_mean_abs_all`: v001-style magnitude ranking.
2. `wanda_decoder_l2_all`: Wanda-style decoder-norm times activation-norm ranking.
3. `final_token_mean_abs`: position-aware final-token activation ranking.
4. `final_token_wanda`: final-token Wanda-style ranking.


In [ ]:
def run_baseline_comparison(
    k_values,
    dataset,
    sae,
    layer,
    rankings,
    batch_size=BATCH_SIZE,
    preserve_error=True,
    output_csv_path=None,
):
    texts = dataset["clean_prompt"].to_list()
    full_logits = run_logits(texts, batch_size=batch_size, use_sae_substitution=False)
    full_logit_diff = mean_logit_diff(full_logits, dataset)

    all_mask = torch.ones(int(sae.cfg.d_sae), device=device)
    zero_mask = torch.zeros(int(sae.cfg.d_sae), device=device)
    reconstruction_metrics = compute_faithfulness(
        sae, layer, all_mask, dataset, full_logit_diff=full_logit_diff, batch_size=batch_size, preserve_error=preserve_error
    )
    zero_metrics = compute_faithfulness(
        sae, layer, zero_mask, dataset, full_logit_diff=full_logit_diff, batch_size=batch_size, preserve_error=preserve_error
    )
    print("All-feature metrics:", reconstruction_metrics)
    print("Zero-feature metrics:", zero_metrics)

    rows = []
    for baseline_name, top_features in rankings.items():
        for k in tqdm(k_values, desc=f"{baseline_name} sweep"):
            mask = make_topk_mask(top_features, int(k), d_sae=int(sae.cfg.d_sae))
            metrics = compute_faithfulness(
                sae=sae,
                layer=layer,
                mask=mask,
                dataset=dataset,
                full_logit_diff=full_logit_diff,
                batch_size=batch_size,
                preserve_error=preserve_error,
            )
            rows.append(
                {
                    "baseline": baseline_name,
                    "k": int(k),
                    "layer": int(layer),
                    "preserve_error": bool(preserve_error),
                    **metrics,
                }
            )

    results = pd.DataFrame(rows)
    if output_csv_path is not None:
        results.to_csv(output_csv_path, index=False)
        print(f"Saved baseline comparison results to {output_csv_path}")
    return results, reconstruction_metrics, zero_metrics


def plot_baseline_comparison(results, reconstruction_metrics=None, zero_metrics=None, output_path=None, title="SAE feature ranking comparison"):
    fig, ax = plt.subplots(figsize=(8.5, 5.2))
    for baseline_name, group in results.groupby("baseline"):
        ordered = group.sort_values("k")
        ax.plot(ordered["k"], ordered["faithfulness"], marker="o", linewidth=1.6, label=baseline_name)

    ax.set_xscale("log")
    ax.set_xlabel("Circuit size (active SAE features)")
    ax.set_ylabel("Faithfulness (masked logit diff / full logit diff)")
    ax.set_title(title)
    ax.grid(True, which="both", alpha=0.3)
    ax.axhline(1.0, color="gray", linestyle=":", linewidth=1.2, label="Full model / all features")

    if reconstruction_metrics is not None:
        ax.axhline(
            reconstruction_metrics["faithfulness"],
            color="tab:orange",
            linestyle="--",
            linewidth=1.1,
            label="All-feature SAE intervention",
        )
    if zero_metrics is not None:
        ax.axhline(
            zero_metrics["faithfulness"],
            color="tab:red",
            linestyle="--",
            linewidth=1.1,
            label="Zero-feature SAE intervention",
        )

    ax.legend(fontsize=8)
    fig.tight_layout()
    assert ax.get_xscale() == "log"
    if output_path is not None:
        fig.savefig(output_path, dpi=180, bbox_inches="tight")
        print(f"Saved plot to {output_path}")
    plt.show()
    return fig, ax


## 10. Smoke Test

Run this first. It validates the v2 ranking and error-preserving masking on 8 prompts.


In [ ]:
smoke_stats = cache_sae_feature_stats(
    smoke_dataset,
    target_sae,
    TARGET_LAYER,
    cache_path=SMOKE_STATS_CACHE_PATH,
    batch_size=BATCH_SIZE,
    force_recompute=True,
)
smoke_scores, smoke_rankings = make_feature_rankings(smoke_stats, target_sae)
smoke_results, smoke_reconstruction, smoke_zero = run_baseline_comparison(
    SMOKE_K_VALUES,
    smoke_dataset,
    target_sae,
    TARGET_LAYER,
    smoke_rankings,
    batch_size=BATCH_SIZE,
    preserve_error=PRESERVE_ERROR_FOR_SWEEP,
    output_csv_path=SMOKE_RESULTS_CSV_PATH,
)
write_run_manifest("smoke_completed", extra={"smoke_rows": len(smoke_results)})
smoke_results


In [ ]:
plot_baseline_comparison(
    smoke_results,
    reconstruction_metrics=smoke_reconstruction,
    zero_metrics=smoke_zero,
    output_path=SMOKE_PLOT_PATH,
    title="Smoke test: v002 SAE feature ranking comparison",
)


## 11. Full v002 Baseline Run

This runs the full 1,000-prompt comparison with larger K values. It writes the CSV, plot, and manifest to the versioned Drive trial folder.


In [ ]:
feature_stats = cache_sae_feature_stats(
    full_dataset,
    target_sae,
    TARGET_LAYER,
    cache_path=STATS_CACHE_PATH,
    batch_size=BATCH_SIZE,
    force_recompute=False,
)
feature_scores, feature_rankings = make_feature_rankings(feature_stats, target_sae)

print("Top 10 feature ids by ranking:")
for name, ranking in feature_rankings.items():
    print(name, ranking[:10].detach().cpu().tolist())


In [ ]:
results, reconstruction_metrics, zero_metrics = run_baseline_comparison(
    K_VALUES,
    full_dataset,
    target_sae,
    TARGET_LAYER,
    feature_rankings,
    batch_size=BATCH_SIZE,
    preserve_error=PRESERVE_ERROR_FOR_SWEEP,
    output_csv_path=RESULTS_CSV_PATH,
)
results


In [ ]:
plot_baseline_comparison(
    results,
    reconstruction_metrics=reconstruction_metrics,
    zero_metrics=zero_metrics,
    output_path=PLOT_PATH,
    title="v002: magnitude vs Wanda-style vs final-token SAE ranking",
)
manifest = write_run_manifest(
    "completed",
    extra={
        "rows": len(results),
        "layer_sanity_rows": len(layer_sanity),
        "reconstruction_metrics": reconstruction_metrics,
        "zero_metrics": zero_metrics,
    },
)
copied = mirror_artifacts_to_latest([
    LAYER_SANITY_CSV_PATH,
    SMOKE_RESULTS_CSV_PATH,
    SMOKE_PLOT_PATH,
    RESULTS_CSV_PATH,
    PLOT_PATH,
    MANIFEST_PATH,
])
print("Completed run manifest:")
print(json.dumps(manifest, indent=2))
print("Mirrored latest artifacts:")
for path in copied:
    print(path)


## 12. Expected Artifacts

After the full run, Google Drive should contain:

```text
MyDrive/minimum_sae_circuit_discovery/
  cache/v002_wanda_position_sanity/
    feature_stats_layer8_ioi.pt
    smoke_feature_stats_layer8_ioi.pt
  runs/v002_wanda_position_sanity/trial_YYYYMMDD_HHMMSS/
    run_manifest.json
    layer_reconstruction_sanity.csv
    smoke_baseline_comparison_results.csv
    smoke_baseline_comparison.png
    baseline_comparison_results.csv
    baseline_comparison.png
  latest/v002_wanda_position_sanity/
    run_manifest.json
    layer_reconstruction_sanity.csv
    baseline_comparison_results.csv
    baseline_comparison.png
```

Interpretation guide:

- If `preserve_error=False` has low all-feature faithfulness but `preserve_error=True` is near 1.0, v001 was bottlenecked by SAE reconstruction error.
- If Wanda-style rankings beat plain activation magnitude at small K, activation-weighted/decoder-weighted feature selection is the stronger next baseline.
- If final-token rankings win, the next search method should operate on feature-position pairs, not only global feature IDs.
